In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import argparse
from nnfabrik.builder import get_data, get_trainer

from modified_neuralpredictors.create_model import stacked_core_full_gauss_readout
from modified_neuralpredictors.wandb_trainer import standard_trainer
from modified_neuralpredictors.dataloaders import static_loaders
from autoencoder import Autoenc

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = 'cuda:7'
torch.cuda.set_device(device)

basepath = "/srv/user/polina/sensorium/sensorium/notebooks/data/"

# as filenames, we'll select all 7 datasets
filenames = [
    os.path.join(basepath, file) for file in os.listdir(basepath) if ".zip" in file
]

dataset_fn = "sensorium.datasets.static_loaders"
dataset_config = {
    "paths": filenames,
    "normalize": True,
    "include_behavior": True,
    "include_eye_position": True,
    "batch_size": 128,
    "scale": 0.25, 
    #"neuron_base_seed": 0,
}

num_val_neurons = 500
''' 
train_config = dataset_config.copy()
train_config['exclude_neuron_n'] = num_val_neurons
trainloaders = static_loaders(**train_config)

val_config = dataset_config.copy()
val_config["neuron_n"] = num_val_neurons
valloaders = static_loaders(**val_config)

data_keys = list(trainloaders['train'].keys())
'''

dataloaders = get_data(dataset_fn, dataset_config)

In [2]:
model_config = {
    'hidden_channels': 128, # original sensorium was 64ch
    'depth_separable': False,
    'use_avg_reg': False,
    'laplace_padding': None,
    'momentum': 0.75,
    'final_nonlinearity': True,
    'nonlinearity_type': 'AdaptiveELU',
    'pad_input': False,
    'stack': -1,
    'layers': 4,
    'input_kern': 11, #  original sensorium was 'input_kern': 9,
    'gamma_input': 6.3831, # this should not influence the performance based on previous experience
    
    'feature_reg_weight': 3, # this is the one I actually would like to tune!
    
    'hidden_kern': 7,
    'grid_mean_predictor': {
        'type': 'cortex',
        'input_dimensions': 2,
        'hidden_layers': 1,
        'hidden_features': 30,
        'final_tanh': True
    },
    
    'init_sigma': 0.1,
    'init_mu_range': 0.3,
    'gauss_type': 'full',
    'shifter': True,
    'batch_norm_scale': [True, True, True, False],
    'core_bias': [True, True, True, False],
    'regularizer_type': "adaptive_log_norm",
    'gamma_sigma' : 0.25,
}

model = stacked_core_full_gauss_readout(dataloaders, random_seed, **model_config)
model.load_state_dict(torch.load('runs/base/exp0/model_weights.pth'))

/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:74: UserWarning: Use of 'gamma_readout' is deprecated. Use 'feature_reg_weight' instead. If 'feature_reg_weight' is defined, 'gamma_readout' is ignored
  warnings.warn(
/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:95: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


<All keys matched successfully>

In [3]:
'''
# select all parameters that are not autoencoder
freeze_params = ['core', 'readout', 'shifter']

# copy parameters and freeze them
for name, param in model.named_parameters():
    param.requires_grad = not any(word in name for word in freeze_params)
'''
for param in model.parameters():
    param.requires_grad = False

In [4]:
autoencoder = Autoenc(
    input_dim=128, 
    latent_dim=128, 
    hidden_layers=0, 
    hidden_dims=128, 
    batch_norm=False, 
    nonlinearity='GELU'
)
data_keys = list(dataloaders['train'].keys())
for data_key in data_keys:
    model.readout[data_key].autoencoder = autoencoder

In [5]:
[(name, param.shape, param.requires_grad) for name, param in model.named_parameters()]

[('core.features.layer0.conv.weight', torch.Size([128, 4, 11, 11]), False),
 ('core.features.layer0.norm.weight', torch.Size([128]), False),
 ('core.features.layer0.norm.bias', torch.Size([128]), False),
 ('core.features.layer1.conv.weight', torch.Size([128, 128, 7, 7]), False),
 ('core.features.layer1.conv.bias', torch.Size([128]), False),
 ('core.features.layer1.norm.weight', torch.Size([128]), False),
 ('core.features.layer1.norm.bias', torch.Size([128]), False),
 ('core.features.layer2.conv.weight', torch.Size([128, 128, 7, 7]), False),
 ('core.features.layer2.conv.bias', torch.Size([128]), False),
 ('core.features.layer2.norm.weight', torch.Size([128]), False),
 ('core.features.layer2.norm.bias', torch.Size([128]), False),
 ('core.features.layer3.conv.weight', torch.Size([128, 128, 7, 7]), False),
 ('core.features.layer3.conv.bias', torch.Size([128]), False),
 ('core.features.layer3.norm.weight', torch.Size([128]), False),
 ('core.features.layer3.norm.bias', torch.Size([128]), Fal

In [6]:
''' 
core_lr = 1e-10
autoencoder_lr = 1e-10
readout_lr = 1e-5
shifter_lr = 1e-5

optimizer = torch.optim.Adam([
    #{'params': model_autoenc.core.parameters(), 'lr': core_lr}, 
    {'params': model.readout[data_key].autoencoder.parameters(), 'lr': autoencoder_lr},
    #{'params': model_autoenc.readout[data_key].sigma, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key]._features, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key].bias, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key].mu_transform.parameters(), 'lr': readout_lr},
    #{'params': model_autoenc.shifter.parameters(), 'lr': shifter_lr},
])
'''

# Neuron idcs train/test splits
num_val_neurons = 500
neuron_idcs = {
    'train': {}, 
    'validation': {}, 
}
data_keys = list(dataloaders['train'].keys())

for key in data_keys:
    responses = next(iter(dataloaders['train'][key]))[1]
    num_neurons = responses.shape[1]
    #print(num_neurons)
    
    idcs = np.random.permutation(num_neurons)
    neuron_idcs['train'][key] = idcs[num_val_neurons:]
    neuron_idcs['validation'][key] = idcs[:num_val_neurons]

trainer_config = {
    'max_iter': 200,
    'verbose': False,
    'lr_decay_steps': 4,
    'avg_loss': False,
    'lr_init': 0.009,
    'device': device,
    'wandb_project': 'small readout vectors',
    'wandb_config': '',
    'wandb_name': 'small autoencoder test', 
    'train_neurons': neuron_idcs['train'], 
    'validation_neurons': neuron_idcs['validation'],  
}

validation_score, trainer_output, state_dict = standard_trainer(
    model, 
    dataloaders, 
    seed=42, 
    **trainer_config
)

#torch.save(autoencoder.state_dict(), 'autoencoder_weights.pth')

/user/nathanpaul.soeding/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: nathan-soeding (ecker-lab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 33: 100%|██████████| 252/252 [00:37<00:00,  6.66it/s]


final/validation_corr_all,▁
final/validation_corr_mean,▁
val/correlation,▁████████████████████████████████
val/poisson_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
final/validation_corr_all,0.34777
final/validation_corr_mean,0.34777
val/correlation,0.36184
val/poisson_loss,15363047.0


In [8]:
trainer = get_trainer('sensorium.training.standard_trainer', trainer_config)

In [10]:
trainer(model, dataloaders, seed=42)

/user/nathanpaul.soeding/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1:   9%|▉         | 23/252 [00:08<01:24,  2.71it/s]


KeyboardInterrupt: 

In [10]:
validation_score

np.float32(0.3247073)

In [12]:
torch.save(autoencoder.state_dict(), 'autoencoder_weights.pth')

In [13]:
autoencoder.load_state_dict(torch.load('autoencoder_weights.pth'))

<All keys matched successfully>

In [ ]:
from sensorium.utility.scores import get_correlations

for data_key in data_keys:
    model.readout[data_key].autoencoder = None

model.eval()

# Compute avg validation and test correlation
validation_correlation = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)

print(validation_correlation)

0.1445416
